# Mini-Solveit

In [ ]:
import bleach
import json
import fasthtml.components as fh
import mistletoe

from claudette import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
from toolslm.shell import get_shell

In [ ]:
MARKDOWN_ALLOWED_TAGS = bleach.ALLOWED_TAGS | {
    'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
    'p', 'pre', 'hr', 'br', 'img'
}

def md_to_html(text):
    return NotStr(bleach.clean(mistletoe.markdown(text), tags=MARKDOWN_ALLOWED_TAGS))

In [ ]:
def get_preview(app): return partial(HTMX, app=app, host=None, port=None)

In [ ]:
def mk_compfn(name, compcls, tag=None):
    if not tag: tag = name
    compfunc = getattr(fh, tag)
    def fn(*c, xtra= '', cls='', **kw):
        xtra = " ".join(f"{compcls}-{o}" for o in xtra.split())
        return compfunc(*c, cls=f'{compcls} {xtra} {cls}', **kw)
    fn.__name__ = name
    globals()[name] = fn



In [ ]:
mk_compfn('Button', 'btn')
mk_compfn('ChatC', 'chat', 'Div') # Call it ChatC as to not confuse with claudette Chat class
mk_compfn('Bubble', 'chat-bubble', 'Div')
mk_compfn('Input', 'input')
mk_compfn('TabInput', 'tab', 'Input')

### Set up the app and start server

In [ ]:
daisy_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4')
)

app = FastHTML(hdrs=daisy_hdrs, session_cookie="my_cookie")
rt = app.route

In [ ]:
p = get_preview(app)

In [ ]:
srv = JupyUvi(app)

## New features

- Different tabs for code submission and prompt submission
- Build route for prompt submission `execprompt`
- Track history so LLM sees everything the human does
- Multi-user support via session cookies
- Database backend
- Refresh/run all when dialog is opened
- Home page that lists existing dialogs
- User can create a new dialog
- Back button to return to dialog list
- Render AI responses as markdown

In [ ]:
def qa_bubble(q, a, role):
    return Div(
        ChatC(Bubble(str(q)), xtra="start"),
        ChatC(Bubble(str(a) if role=='computer' else md_to_html(a), xtra=f"{'primary' if role=='computer' else 'success'}"), xtra="end") if a else None
    )

p(qa_bubble('2+2', '4', role='ai'))

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_ibhhAAL9Sy61B5jsP-RJQw&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;
      &lt;div class=&quot;chat chat-start &quot;&gt;
        &lt;div class=&quot;chat-bubble  &quot;&gt;2+2&lt;/div&gt;
      &lt;/div&gt;
      &lt;div class=&quot;chat chat-end &quot;&gt;
        &lt;div class=&quot;chat-bubble chat-bubble-success &quot;&gt;&lt;p&gt;4&lt;/p&gt;
&lt;/div&gt;
      &lt;/div&gt;
    &lt;/div&gt;
  &lt;/body&gt;
&lt;/html&gt;
" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
def ex(c, sh):
    res = sh.run_cell(c)
    return res.result if res.result is not None else res.stdout

In [ ]:
def msgs_to_chat_history(msgs):
    history = []
    code = []
    for msg in msgs:
        if msg['interaction'] == 'computer': code.append(f"[code cell]:{msg['prompt']}\n[output]:{msg['answer']}")
        else:
            user_content = (f"{'\n'.join(code)}\n{msg['prompt']}" if code else msg['prompt'])
            code = []
            history.append({'role': 'user', 'content': user_content})
            history.append({'role': 'assistant', 'content': [{'text': msg['answer'], 'type': 'text'}]})
    return history, code

In [ ]:
def exprompt(msgs, prompt):
    h, code = msgs_to_chat_history(msgs)
    c = Chat(model="claude-haiku-4-5", sp="You are a helpful coding assistant who answers concisely.", hist=h)
    c(f"{'\n'.join(code)}\n{prompt}" if code else prompt)
    return c.last[0].content[0].text

In [ ]:
@rt
def execcode(code:str, dlgid:int, session):
    assert 'uid' in session
    
    # Execute code
    sh = udlg_active[dlgid]
    result = ex(code, sh)
    
    # Track message history
    msgs = json.loads(dialogs[dlgid].messages)
    msgs.append({'prompt': code, 'answer': result, 'interaction': 'computer'})
    dialogs.update(dict(id=dlgid, messages=msgs))
    
    return qa_bubble(code, result, role="computer"), Input(placeholder='Enter code here', id='code', hx_swap_oob='outerHTML',cls="flex-1")

In [ ]:
@rt
def execprompt(prompt:str, dlgid:int, session):
    assert 'uid' in session

    msgs = json.loads(dialogs[dlgid].messages)    
    answer = exprompt(msgs, prompt)

    # Track message history
    msgs.append({'prompt': prompt, 'answer': answer, 'interaction': 'ai'})
    dialogs.update(dict(id=dlgid, messages=msgs))
    
    return qa_bubble(prompt, answer, role="ai"), Input(placeholder='Enter prompt here', id='prompt', hx_swap_oob='outerHTML',cls="flex-1")

### DB support

In [ ]:
db = database('mini-solveit.db')

class User: id:str
class Dialog: id:int; uid:int; messages:str; name:str

users = db.create(User, transform=True)
dialogs = db.create(Dialog, transform=True)

In [ ]:
udlg_active = {}

In [ ]:
def run_all(code_msgs):
    sh = get_shell()
    for code in code_msgs: ex(code, sh)
    return sh

In [ ]:
@rt
def showdlg(session, dlgid:int):
    if 'uid' not in session:
        session['uid'] = unqid()
    uid = session['uid']
    
    msgs = json.loads(dialogs[dlgid].messages)
    code_msgs = [msg['prompt'] for msg in msgs if msg['interaction']=='computer']
    if dlgid not in udlg_active: udlg_active[dlgid] = run_all(code_msgs)

    return Div(cls="flex flex-col h-screen max-w-lg mx-auto pt-4")(
        A("← Back", href="/dlglist", cls="btn w-fit"),
        H1(dialogs[dlgid].name, cls="text-3xl text-center font-bold"),
        Div(*[qa_bubble(msg['prompt'], msg['answer'], msg['interaction']) for msg in msgs], id="dest", cls="flex-1 overflow-y-auto"),
        Div(cls="tabs tabs-lift sticky bottom-0 bg-base-100")(
            # Code tab
            TabInput(type="radio", name="mode", aria_label="Code", checked=True),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execcode", hx_target="#dest", hx_swap="beforeend scroll:#dest:bottom")(
                    Hidden(value=dlgid, name="dlgid"),
                    Input(placeholder="Enter code here", id="code", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
            # Prompt tab
            TabInput(type="radio", name="mode", aria_label="Prompt"),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execprompt", hx_target="#dest", hx_swap="beforeend scroll:#dest:bottom")(
                    Hidden(value=dlgid, name="dlgid"),
                    Input(placeholder="Enter prompt here", id="prompt", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
        )
    )

In [ ]:
p(showdlg.to(dlgid=3))

HTML(<iframe src="/showdlg?dlgid=3" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
@rt
def newdlg(name:str, session):
    assert 'uid' in session
    d = dialogs.insert(uid=session['uid'], messages=json.dumps([]), name=name)
    return Redirect(showdlg.to(dlgid=d.id))

In [ ]:
@rt
def dlglist(session):
    if 'uid' not in session:
        session['uid'] = unqid()
    uid = session['uid']
    try: user = users[uid]
    except NotFoundError: user = users.insert(id=uid)
    udlgs = dialogs('uid=?', [user.id])
    return Div(cls="px-2")(
        H1("Create a new dialog"),
        Div(cls="p-5")(
            Form(hx_get="/newdlg")(
                Input(placeholder="Enter new dialog name", name="name", cls="flex-1"),
                Button("Create", xtra="primary")
            )
        ),
        Div(
            H1("User dialogs"),
            Ul(*[Li(A(dlg.name, href=showdlg.to(dlgid=dlg.id), cls="link"), cls="pl-4") for dlg in udlgs], cls="list space-y-2")
        )
    )

In [ ]:
p(dlglist)

HTML(<iframe src="/dlglist" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
import base64


In [ ]:
cookie_text = "eyJ1aWQiOiAiX2pfU2ZGLXhhUU51VWg5SlVfU2o0Y0EifQ==.aeJFXA.sw1xppOpz36lyr4qlL_n41oQuCQ".split('.')[0]
base64.b64decode(cookie_text)

b'{"uid": "_j_SfF-xaQNuUh9JU_Sj4cA"}'